# 02b Train HR proxies and calibrate detectors - folds 2-5

Attach hcsinhgoethe/ppg-cvae-prepared-folds02-05. Run on T4.
Each fold uses its own train/validation arrays, proxy checkpoint and locked detector.
Outputs: artifacts/proxy/fold_02..fold_05. Test is not read.
Review quality per fold before tuning.


In [1]:
from pathlib import Path
import os, sys, platform, re, torch

print("=== CHECKING /kaggle/input ===")
input_dir = Path("/kaggle/input")
if input_dir.exists():
    items = list(input_dir.iterdir())
    print(f"Contents of /kaggle/input ({len(items)} items): {[p.name for p in items]}")
    for it in items:
        if it.is_dir():
            print(f"  Folder {it.name} contains {len(list(it.iterdir()))} items")
else:
    print("/kaggle/input does not exist!")

# Robust discovery of DATA_ROOT
DATA_ROOT = None
if os.environ.get('PPGDALIA_ROOT'):
    p = Path(os.environ['PPGDALIA_ROOT'])
    if p.exists():
        DATA_ROOT = p

if DATA_ROOT is None and input_dir.exists():
    for p in input_dir.rglob("*.pkl"):
        if re.search(r'(?<![A-Z0-9])S(1[0-5]|[1-9])(?![A-Z0-9])', p.stem.upper()):
            rel = p.relative_to(input_dir)
            DATA_ROOT = input_dir / rel.parts[0]
            print(f"Auto-discovered DATA_ROOT via subject files: {DATA_ROOT}")
            break

if DATA_ROOT is None:
    _candidates = [
        Path('/kaggle/input/ppg-dalia-dataset'),
        Path('/kaggle/input/ppg-dalia'),
        Path('/kaggle/input/datasets/ameersifat53/ppg-dalia-dataset')
    ]
    DATA_ROOT = next((p for p in _candidates if p.exists() and p.is_dir()), _candidates[0])

# Project-relative output is collected by krun.yaml outputs: artifacts/.
OUTPUT_ROOT = Path(os.environ.get('PPGCVAE_OUTPUT_ROOT', 'artifacts'))
FOLDS = (2, 3, 4, 5)
SEED = 42
print({'DATA_ROOT': str(DATA_ROOT), 'OUTPUT_ROOT': str(OUTPUT_ROOT), 'device': 'cuda' if torch.cuda.is_available() else 'cpu', 'python': sys.version, 'torch': torch.__version__})
# This stage consumes prepared arrays/checkpoints; DATA_ROOT is not required.


=== CHECKING /kaggle/input ===
Contents of /kaggle/input (1 items): ['datasets']
  Folder datasets contains 1 items


{'DATA_ROOT': '/kaggle/input/ppg-dalia-dataset', 'OUTPUT_ROOT': 'artifacts', 'device': 'cuda', 'python': '3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]', 'torch': '2.11.0+cu128'}


In [2]:
# Locate this repository without assuming a Kaggle dataset slug.
repo_candidates = [Path.cwd(), *Path.cwd().parents, Path.cwd() / 'Final_IOT_for_AI']
repo_root = next((p for p in repo_candidates if (p / 'src').exists()), None)
if repo_root is None:
    raise FileNotFoundError('Could not locate repository src/ directory; upload/clone the repository first.')
sys.path.insert(0, str(repo_root / 'src'))
from ppg_cvae.data.splits import get_fold
for fold in FOLDS:
    print(get_fold(fold))


def import_prepared_fold(FOLD):
    # Prepared dataset: https://www.kaggle.com/datasets/hcsinhgoethe/ppg-cvae-prepared-folds02-05
    import shutil
    from zipfile import ZipFile
    DATASET_SLUG = 'ppg-cvae-prepared-folds02-05'
    ARTIFACT_INPUT_ROOT = os.environ.get('PPGCVAE_INPUT_ROOT')
    fold_name = f'fold_{FOLD:02d}'
    required_files = ('train.npz', 'val.npz', 'manifest.csv', 'normalization_stats.json', 'preprocessing_config.json')
    destination = OUTPUT_ROOT/'prepared'/fold_name

    # Discover the mounted dataset rather than assuming Kaggle's directory layout.
    if ARTIFACT_INPUT_ROOT:
        dataset_roots = [Path(ARTIFACT_INPUT_ROOT)]
    else:
        dataset_roots = sorted(p for p in input_dir.rglob(DATASET_SLUG) if p.is_dir()) if input_dir.exists() else []
    if not dataset_roots:
        raise FileNotFoundError(f'Attach hcsinhgoethe/{DATASET_SLUG} with krun --dataset, or set PPGCVAE_INPUT_ROOT.')

    prepared_sources = []
    for root in dataset_roots:
        if not root.is_dir(): raise FileNotFoundError(root)
        for train_file in root.rglob('train.npz'):
            candidate = train_file.parent
            if candidate.name.startswith('fold_') and candidate.name != fold_name: continue
            if all((candidate/name).is_file() for name in required_files):
                prepared_sources.append(candidate)
    if len(prepared_sources) > 1:
        raise ValueError(f'Ambiguous prepared artifact directories: {prepared_sources}; set PPGCVAE_INPUT_ROOT explicitly.')
    if prepared_sources:
        prepared_source = prepared_sources[0]
        if prepared_source.resolve() != destination.resolve():
            shutil.copytree(prepared_source, destination, dirs_exist_ok=True)
        print('Imported prepared arrays from:', prepared_source)
    else:
        # Also support a dataset whose uploaded ZIP has not been unpacked by Kaggle.
        archives = []
        for root in dataset_roots:
            for archive in root.rglob('*.zip'):
                with ZipFile(archive) as z:
                    prefix = f'prepared/{fold_name}/'
                    if all(prefix+name in z.namelist() for name in required_files):
                        archives.append(archive)
        if len(archives) != 1:
            raise FileNotFoundError(f'Expected one prepared {fold_name} directory or ZIP in {dataset_roots}; found archives: {archives}')
        destination.mkdir(parents=True, exist_ok=True)
        with ZipFile(archives[0]) as z:
            prefix = f'prepared/{fold_name}/'
            for name in (*required_files, 'test.npz', 'subject_summary.csv', 'hr_coverage.csv'):
                member = prefix+name
                if member in z.namelist():
                    with z.open(member) as src, (destination/name).open('wb') as dst:
                        shutil.copyfileobj(src, dst)
        print('Imported prepared arrays from ZIP:', archives[0])
    assert all((destination/name).is_file() for name in required_files)
    print('Prepared fold directory:', destination)
    return destination


FoldSubjects(fold_id=2, train_subjects=frozenset({'S11', 'S3', 'S14', 'S13', 'S10', 'S15', 'S1', 'S12', 'S2'}), val_subjects=frozenset({'S7', 'S9', 'S8'}), test_subjects=frozenset({'S6', 'S4', 'S5'}))
FoldSubjects(fold_id=3, train_subjects=frozenset({'S6', 'S3', 'S4', 'S14', 'S13', 'S15', 'S1', 'S5', 'S2'}), val_subjects=frozenset({'S11', 'S10', 'S12'}), test_subjects=frozenset({'S7', 'S9', 'S8'}))
FoldSubjects(fold_id=4, train_subjects=frozenset({'S6', 'S3', 'S4', 'S7', 'S9', 'S8', 'S1', 'S5', 'S2'}), val_subjects=frozenset({'S13', 'S14', 'S15'}), test_subjects=frozenset({'S11', 'S10', 'S12'}))
FoldSubjects(fold_id=5, train_subjects=frozenset({'S6', 'S11', 'S4', 'S7', 'S10', 'S9', 'S8', 'S12', 'S5'}), val_subjects=frozenset({'S1', 'S3', 'S2'}), test_subjects=frozenset({'S13', 'S14', 'S15'}))


In [3]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device != 'cuda':
    raise RuntimeError('Run KRun with --gpu T4 for the four-fold proxy job.')
for FOLD in FOLDS:
    print(f'Starting proxy and detector for fold {FOLD}', flush=True)
    import_prepared_fold(FOLD)
    import numpy as np
    import pandas as pd
    from ppg_cvae.data.prepare import load_prepared
    from ppg_cvae.training.train_hr_proxy import train_proxy
    from ppg_cvae.evaluation.calibration import calibrate_detector
    from ppg_cvae.evaluation.heart_rate import estimate_hr
    prepared_dir = OUTPUT_ROOT/'prepared'/f'fold_{FOLD:02d}'
    train_x, train_c = load_prepared(prepared_dir, 'train')
    val_x, val_c = load_prepared(prepared_dir, 'val')
    proxy_dir = OUTPUT_ROOT/'proxy'/f'fold_{FOLD:02d}'
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    proxy, history = train_proxy(train_x, train_c, val_x, val_c, device=device,
                                 output_dir=proxy_dir, seed=42)
    detector = calibrate_detector(val_x, val_c, proxy_dir)
    with torch.no_grad():
        predictions = torch.cat([proxy(torch.as_tensor(val_x[i:i+128], device=device))
                                 .cpu() for i in range(0,len(val_x),128)]).numpy()*60
    validation = pd.read_csv(prepared_dir/'manifest.csv')
    validation = validation[(validation.split=='val') & validation.valid].copy()
    validation['proxy_hr_bpm'] = predictions
    validation['proxy_abs_error'] = np.abs(predictions-val_c*60)
    measurements = [estimate_hr(x, **detector) for x in val_x]
    validation['peak_hr_bpm'] = [m.hr for m in measurements]
    validation['peak_measurable'] = [m.measurable for m in measurements]
    validation['peak_abs_error'] = np.abs(validation.peak_hr_bpm-val_c*60)
    validation['hr_bin_lower'] = (validation.hr_reference_bpm//20)*20
    validation.to_csv(proxy_dir/'real_validation_measurements.csv', index=False)
    for group in ('subject_id','hr_bin_lower'):
        validation.groupby(group).agg(windows=('subject_id','size'),
            proxy_mae_bpm=('proxy_abs_error','mean'), peak_mae_bpm=('peak_abs_error','mean'),
            peak_V=('peak_measurable','mean')).to_csv(proxy_dir/f'validation_by_{group}.csv')
    print(pd.DataFrame(history).tail())
    print('Locked detector:', detector)
    print('Review proxy error by person and HR before using its HR loss.')
    print(f'Completed proxy and detector for fold {FOLD}', flush=True)
print('Review validation quality for every fold before tuning; test remains closed.')


Starting proxy and detector for fold 2


Imported prepared arrays from: /kaggle/input/datasets/hcsinhgoethe/ppg-cvae-prepared-folds02-05/prepared/fold_02
Prepared fold directory: artifacts/prepared/fold_02


    epoch  train_mae_bpm  val_mae_bpm
45     46       5.512038     7.440675
46     47       5.520109     8.097565
47     48       5.525563     7.668000
48     49       5.512877     8.149319
49     50       5.470469     7.862189
Locked detector: {'min_distance': 20, 'prominence': 0.25}
Review proxy error by person and HR before using its HR loss.
Completed proxy and detector for fold 2


Starting proxy and detector for fold 3


Imported prepared arrays from: /kaggle/input/datasets/hcsinhgoethe/ppg-cvae-prepared-folds02-05/prepared/fold_03
Prepared fold directory: artifacts/prepared/fold_03


...

    epoch  train_mae_bpm  val_mae_bpm
45     46       6.872471     6.067380
46     47       6.877968     6.178718
47     48       6.794954     6.103700
48     49       6.806198     6.206510
49     50       6.752520     6.108667
Locked detector: {'min_distance': 20, 'prominence': 0.5}
Review proxy error by person and HR before using its HR loss.
Completed proxy and detector for fold 4


Starting proxy and detector for fold 5


Imported prepared arrays from: /kaggle/input/datasets/hcsinhgoethe/ppg-cvae-prepared-folds02-05/prepared/fold_05
Prepared fold directory: artifacts/prepared/fold_05


    epoch  train_mae_bpm  val_mae_bpm
45     46       7.472435     7.550830
46     47       7.420697     6.824464
47     48       7.475379     6.788221
48     49       7.488616     7.384965
49     50       7.461263     7.070560
Locked detector: {'min_distance': 20, 'prominence': 0.25}
Review proxy error by person and HR before using its HR loss.
Completed proxy and detector for fold 5


Review validation quality for every fold before tuning; test remains closed.
